<a href="https://colab.research.google.com/github/emrah1982/SmartFarmStrawberryDisease/blob/main/StrawberryVision_Colab_Production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍓 Çilek Hastalık + Olgunluk — YOLO26 Eğitimi (Colab Pro+)

Bu notebook **birleşik dataset** (10 sınıf: 7 hastalık + 3 olgunluk + sağlıklı/background) ile YOLO26 eğitir.

## ⚠️ Çalıştırmadan önce — 2 zorunlu adım

**1) Runtime ayarı (Colab Pro+ için en iyi seçim):**
Runtime → Change runtime type →
- Hardware accelerator: **A100 GPU** ⭐ (en hızlı; yoksa L4)
- Runtime shape: **High-RAM** (A100'de `cache='ram'` devreye girer, epoch süresi düşer)
- Runtime menüsünde **Background execution** açık olsun → tarayıcıyı kapatsanız bile eğitim sürer

**2) Dataset'i Drive'a yükleyin (bir kez):** Dataset GitHub deposunda **yoktur** (399 MB).
Bilgisayarınızdaki `dataset_colab.zip` dosyasını Drive'da şu klasöre yükleyin:

```
MyDrive/SmartFarmStrawberryDisease/dataset/dataset_colab.zip
```

Sonra **Runtime → Run all**. Tek elle müdahale: Drive bağlama hücresi bir kez
yetki onayı ister (Colab'ın güvenlik gereği, atlanamaz).

## 📂 Dosyalar modele nasıl veriliyor?
Görüntüler tek klasörde toplanmaz. `configs/strawberry_data.yaml` içindeki **dizin listesi**
ile 4 kaynak + augment çıktısı birlikte okunur. Label'lar, görüntü yolundaki `/images/` →
`/labels/` değişimiyle otomatik bulunur.

| Split | Görüntü | İçerik |
|---|---|---|
| train | 9.343 | 4 kaynak + augment (200'ü sağlıklı/background) |
| val | 1.341 | sadece orijinal kaynaklar (augment YOK) |
| test | 515 | sadece orijinal kaynaklar |

---

## 1️⃣ Paket kurulumu ve uyumluluk kontrolü

In [1]:
# Colab'da SADECE ultralytics kurulur.
# NEDEN: Colab'da torch / numpy / opencv zaten kurulu ve birbiriyle uyumludur.
# Bunları elle kurmak veya yükseltmek ikili (ABI) uyumsuzluğu yaratır
# ("numpy.dtype size changed", "cv2 import error") ve runtime restart gerektirir.
# ultralytics eksik bağımlılıklarını uyumlu sürümlerle kendisi çeker.
!pip install -q "ultralytics>=8.3.200"

print('\n--- Sürüm / donanım kontrolü ---')
problem = False
try:
    import numpy, torch, cv2, ultralytics, psutil
    print('numpy      :', numpy.__version__)
    print('torch      :', torch.__version__)
    print('opencv     :', cv2.__version__)
    print('ultralytics:', ultralytics.__version__)

    v = tuple(int(x) for x in ultralytics.__version__.split('.')[:3])
    if v < (8, 3, 200):
        print('\n⚠️ ultralytics sürümü YOLO26 için eski: !pip install -U ultralytics')
        problem = True

    ram = psutil.virtual_memory().total / 1e9
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'\n✅ GPU: {name} ({vram:.0f} GB VRAM) | RAM: {ram:.0f} GB')

        up = name.upper()
        if 'A100' in up:
            print('🏆 En hızlı seçenek aktif.')
        elif 'L4' in up:
            print('👍 L4 iyi bir seçim. Daha hızlısı için: Runtime > Change runtime type > A100 GPU')
        else:
            print(f'ℹ️ Pro+ ile daha hızlısı mümkün: Runtime > Change runtime type > A100 GPU')

        if ram < 60:
            print('💡 Runtime shape: High-RAM seçerseniz A100\'de cache=ram açılır ve eğitim hızlanır.')
    else:
        print('\n⚠️ GPU YOK! Runtime > Change runtime type > A100 GPU seçin, sonra bu hücreyi tekrar çalıştırın.')
        problem = True
except Exception as e:
    print('\n❌ Import hatası:', e)
    print('💡 Çözüm: Runtime > Restart session, sonra bu hücreyi TEKRAR çalıştırın.')
    problem = True

print('\n' + ('⚠️ Yukarıdaki uyarıyı giderin' if problem else '✅ Ortam hazır'))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.2 MB/s eta 0:00:00

--- Sürüm / donanım kontrolü ---
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
numpy      : 2.0.2
torch      : 2.11.0+cu128
opencv     : 4.13.0
ultralytics: 8.4.106

✅ GPU: Tesla T4 (16 GB VRAM) | RAM: 14 GB
ℹ️ Pro+ ile daha hızlısı mümkün: Runtime > Change runtime type > A100 GPU
💡 Runtime shape: High-RAM seçerseniz A100'de cache=ram açılır ve eğitim hızlanır.

✅ Ortam hazır


## 2️⃣ Google Drive bağlantısı

Bu hücre bir kez **yetki onayı** ister (açılan pencereden hesabınızı seçin).

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/SmartFarmStrawberryDisease')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
RESULTS_DIR = DRIVE_ROOT / 'results'
MODELS_DIR = DRIVE_ROOT / 'best_models'
for d in (DRIVE_ROOT, CHECKPOINT_DIR, RESULTS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('✅ Drive hazır:', DRIVE_ROOT)

## 3️⃣ Depoyu indir / güncelle

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/emrah1982/SmartFarmStrawberryDisease.git'
REPO_DIR = Path('/content/SmartFarmStrawberryDisease')

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=False)
else:
    os.chdir('/content')
    subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir(REPO_DIR)

print('CWD:', Path.cwd())
assert (Path.cwd() / 'configs' / 'strawberry_data.yaml').exists(), \
    'configs/strawberry_data.yaml yok! Depo doğru klonlanmamış olabilir.'
print('✅ Depo hazır')

## 🔍 Drive kontrolü — dosya yolu doğru mu?

Aşağıdaki hücre Drive'ınızdaki klasörleri ve zip dosyalarını listeler.
Hiçbir şeyi değiştirmez; sadece zip'i doğru yere yükleyip yüklemediğinizi gösterir.

In [ ]:
# 🔍 DRIVE TARAYICI — dosya yolunu doğrulamak için
# Bu hücre hiçbir şeyi değiştirmez; sadece Drive'da NE olduğunu gösterir.
# Zip'i doğru klasöre yükleyip yüklemediğinizi buradan görebilirsiniz.
import os
from pathlib import Path

MYDRIVE = Path('/content/drive/MyDrive')
ZIP_NAME = 'dataset_colab.zip'


def listele(d, baslik, limit=25):
    print(f'\n📂 {baslik}')
    print(f'   {d}')
    if not d.exists():
        print('   ❌ BU KLASÖR YOK')
        return
    items = sorted(d.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower()))
    if not items:
        print('   (boş)')
    for p in items[:limit]:
        if p.is_dir():
            print(f'   📁 {p.name}/')
        else:
            try:
                print(f'   📄 {p.name}   ({p.stat().st_size/1e6:.1f} MB)')
            except OSError:
                # Drive'da kisayol/cop kutusu girdileri listelenir ama acilamaz
                print(f'   📄 {p.name}   (boyut okunamadi — kisayol olabilir)')
    if len(items) > limit:
        print(f'   ... +{len(items)-limit} öğe daha')


print('=' * 62)
print('DRIVE İÇERİĞİ')
print('=' * 62)
listele(MYDRIVE, 'MyDrive (kök)')
listele(DRIVE_ROOT, 'Proje klasörü (DRIVE_ROOT)')
listele(DRIVE_ROOT / 'dataset', 'Dataset klasörü — zip burada olmalı')

print('\n' + '=' * 62)
print('ZIP ARAMASI')
print('=' * 62)
beklenen = DRIVE_ROOT / 'dataset' / ZIP_NAME
print(f'Beklenen konum : {beklenen}')
print('Durum          : ' + ('✅ VAR' if beklenen.exists() else '❌ YOK'))

# MyDrive'da 4 seviyeye kadar tüm zip'leri listele (yanlış klasöre yüklendiyse görünür)
bulunan = []
for cur, dirs, files in os.walk(MYDRIVE):
    if len(Path(cur).relative_to(MYDRIVE).parts) >= 4:
        dirs[:] = []
        continue
    for f in files:
        if f.lower().endswith('.zip'):
            p = Path(cur) / f
            try:
                bulunan.append((p, p.stat().st_size / 1e6))
            except OSError:
                continue   # Drive kisayolu/erisilemeyen girdi — atla

if bulunan:
    print("\nDrive'daki .zip dosyaları:")
    for p, mb in sorted(bulunan, key=lambda x: -x[1]):
        isaret = '  ⬅ BU KULLANILACAK' if p.name == ZIP_NAME else ''
        print(f'   {mb:>7.1f} MB   {p}{isaret}')
else:
    print("\n⚠️ Drive'da (4 seviyeye kadar) hiç .zip bulunamadı.")

# Zip yerine klasör olarak yüklenmiş olabilir mi?
for c in (DRIVE_ROOT / 'dataset', DRIVE_ROOT / 'dataset' / 'dataset_colab'):
    try:
        if c.is_dir() and any(q.name == 'augmented_train' or q.name.endswith('.yolo26')
                              for q in c.iterdir()):
            print(f'\n📁 Açılmış dataset klasörü bulundu: {c}')
            print('   (zip olmasa da sonraki hücre bunu kullanabilir)')
    except OSError:
        pass

print('\n' + '=' * 62)
if beklenen.exists() or any(p.name == ZIP_NAME for p, _ in bulunan):
    print('✅ Zip bulundu → sonraki hücreyi çalıştırabilirsiniz.')
else:
    print('❌ dataset_colab.zip yok.')
    print(f'   Yukarıdaki listeye bakın; dosyayı şu klasöre yükleyin:')
    print(f'   {DRIVE_ROOT / "dataset"}')
    print('   Yükleme bittikten sonra bu hücreyi tekrar çalıştırın.')


## 4️⃣ Dataset'i Drive'dan aç ve doğrula

Arşiv önce **yerel diske kopyalanır**, sonra açılır — Drive üzerinden
doğrudan açmak 22.000 küçük dosyada çok yavaştır (~2-3 dk sürer).

> ✅ **Otomatik:** Bu hücreyi atlasanız bile eğitim ve değerlendirme
> hücreleri dataset'i kendisi hazırlar. Hazırsa saniyeler içinde geçer.
> Yerel disk her yeni oturumda silindiği için hazırlık oturum başına bir kez yapılır.

> 💡 Kodu `scripts/prepare_colab_dataset.py` içindedir; depodan `git pull` ile
> geldiği için güncellemeler notebook'u yeniden indirmeden yansır.


In [ ]:
# Dataset hazırlığı DEPODAKİ scripte devredildi.
# NEDEN: Colab sekmesi açıkken hücre kodu önbellekte kalır; dosyayı güncellesek
# bile eski kod çalışır. Script depoda olduğu için yukarıdaki "git pull" adımı
# her çalıştırmada en güncel halini indirir — düzeltmeler anında yansır.
import subprocess, sys
from pathlib import Path

DATASET_DIR = REPO_DIR / 'dataset'

sonuc = subprocess.run(
    [sys.executable, 'scripts/prepare_colab_dataset.py',
     '--drive-root', str(DRIVE_ROOT), '--repo', str(REPO_DIR)],
    cwd=str(REPO_DIR),
)
if sonuc.returncode != 0:
    raise RuntimeError('Dataset hazırlanamadı — yukarıdaki mesaja bakın.')


In [ ]:
# Eğitimden ÖNCE doğrulama: her dizin var mı, görüntü/label eşleşiyor mu?
import yaml
from pathlib import Path

# MUTLAK YOL ZORUNLU: Ultralytics dataset kökünü data.yaml'ın bulunduğu dizinden türetir.
# Göreli yol verilirse kökü kendi DATASETS_DIR'i altında arar → "images not found".
# REPO_DIR üzerinden kurulur ki çalışma dizini değişse bile doğru kalsın.
DATA_YAML_PATH = str((REPO_DIR / 'configs' / 'strawberry_data.yaml').resolve())
cfg = yaml.safe_load(Path(DATA_YAML_PATH).read_text(encoding='utf-8'))
root = Path(cfg.get('path') or Path(DATA_YAML_PATH).parent)
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

names = cfg['names']
print('📁 Config:', DATA_YAML_PATH)
print('🏷️  Sınıflar:', cfg['nc'], '→', list(names.values()) if isinstance(names, dict) else names)
print()

ok = True
for split in ('train', 'val', 'test'):
    entries = cfg.get(split) or []
    entries = [entries] if isinstance(entries, str) else entries
    n_img = n_lbl = n_bg = 0
    for e in entries:
        d = (root / e).resolve()
        if not d.exists():
            print(f'❌ {split}: dizin YOK → {d}'); ok = False; continue
        ld = Path(str(d).replace('/images', '/labels'))
        if not ld.exists():
            print(f'❌ {split}: labels dizini YOK → {ld}'); ok = False; continue
        imgs = [p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS]
        missing = sum(1 for p in imgs if not (ld / f'{p.stem}.txt').exists())
        n_bg += sum(1 for p in imgs if (ld / f'{p.stem}.txt').exists()
                    and (ld / f'{p.stem}.txt').stat().st_size == 0)
        if missing:
            print(f"⚠️ {split}: {missing} görüntünün label'ı yok → {d.parent.parent.name}"); ok = False
        n_img += len(imgs); n_lbl += len(imgs) - missing
    print(f'{split:<6}: {len(entries)} dizin | {n_img:>5} görüntü | {n_lbl:>5} label | {n_bg} background')

print('\n' + ('✅ Dataset eğitime hazır' if ok else '❌ Sorun var — yukarıdaki uyarılara bakın'))

## 5️⃣ Eğitim konfigürasyonu (GPU'ya göre otomatik optimize)

Parametreler ve gerekçeleri `configs/train_config.yaml` içindeki yorumlardadır.
Aşağıdaki hücre **çalışan GPU'yu algılayıp** `batch` / `workers` / `cache` değerlerini
otomatik ayarlar — elle bir şey değiştirmenize gerek yoktur.

| GPU | VRAM | batch (imgsz 1024) | 200 epoch tahmini |
|---|---|---|---|
| **A100** ⭐ | 40 GB | 32 | ~3-4 saat |
| L4 | 24 GB | 16 | ~7-9 saat |
| V100 | 16 GB | 8 | ~10-12 saat |
| T4 | 16 GB | 8 | 15+ saat |

> ⚠️ **GPU tipi koddan seçilemez** — Colab'ın runtime ayarıdır.
> Pro+ ile: **Runtime → Change runtime type → A100 GPU** + **High-RAM** seçin.
> High-RAM açıkken A100 profilinde `cache='ram'` devreye girer: görüntüler belleğe
> alınır, disk okuma darboğazı kalkar ve epoch süresi belirgin düşer.

> 💡 **Pro+ avantajı:** Runtime menüsünden **Background execution**'ı açarsanız
> tarayıcıyı kapatsanız bile eğitim sürer — 200 epoch'u tek oturumda bitirebilirsiniz.

In [ ]:
import yaml, torch, psutil
from pathlib import Path

TRAIN_CONFIG = yaml.safe_load((REPO_DIR / 'configs' / 'train_config.yaml').read_text(encoding='utf-8'))

# --- GPU'ya göre OTOMATİK optimizasyon --------------------------------------
# Colab Pro+ farklı GPU'lar verebilir (A100 40GB / L4 24GB / V100 16GB / T4 16GB).
# batch, VRAM ile doğru orantılı seçilir: çok büyük → CUDA OOM, çok küçük →
# GPU boşta bekler ve eğitim gereksiz uzar. Aşağıdaki değerler yolo26s +
# imgsz 1024 için güvenli üst sınırlardır (~%80 VRAM kullanımı).
PROFILES = {           # (batch @ imgsz1024, workers, açıklama)
    'A100': (32, 12, 'en hızlı — 200 epoch ~3-4 saat'),
    'L4':   (16,  8, 'iyi denge — 200 epoch ~7-9 saat'),
    'V100': ( 8,  8, 'orta — 200 epoch ~10-12 saat'),
    'T4':   ( 8,  8, 'yavaş — 200 epoch 15+ saat, imgsz 640 önerilir'),
}

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
key = next((k for k in PROFILES if k in gpu_name.upper().replace(' ', '')), None)

if key:
    batch, workers, note = PROFILES[key]
    TRAIN_CONFIG['batch'] = batch
    TRAIN_CONFIG['workers'] = workers
    print(f'🎯 GPU: {gpu_name} ({vram:.0f} GB) → {key} profili')
    print(f'   batch={batch}, workers={workers}  ({note})')
else:
    TRAIN_CONFIG['batch'] = max(4, int(vram // 1.3)) if vram else 4
    print(f'ℹ️ Tanınmayan GPU: {gpu_name} ({vram:.0f} GB) → batch={TRAIN_CONFIG["batch"]} (tahmini)')

TRAIN_CONFIG['amp'] = True   # karma hassasiyet; A100'de otomatik bf16

# Görüntüleri RAM'e almak disk okuma darboğazını kaldırır ama ~30 GB ister.
# Yalnızca A100 + High-RAM runtime'da açılır; aksi halde oturum RAM'den çöker.
ram_gb = psutil.virtual_memory().total / 1e9
if key == 'A100' and ram_gb > 60:
    TRAIN_CONFIG['cache'] = 'ram'
    print(f'   cache=ram (RAM {ram_gb:.0f} GB) — veri okuma darboğazı kalkar')
else:
    TRAIN_CONFIG['cache'] = False

# --- Elle geçersiz kılma (isteğe bağlı) -------------------------------------
# Hızlı ilk tur:    {'epochs': 50, 'imgsz': 640, 'batch': 32}
# OOM alırsanız:    {'batch': <yarısı>}
OVERRIDES = {}
TRAIN_CONFIG.update(OVERRIDES)

# Sonuçlar doğrudan Drive'a yazılsın (oturum kopsa bile checkpoint'ler kalır)
TRAIN_CONFIG['project'] = str(RESULTS_DIR)


def find_run_dir():
    """Gerçek koşu dizinini bulur.

    exist_ok=False olduğu için Ultralytics her yeni eğitimde strawberry_exp2,
    strawberry_exp3 ... oluşturur. Sabit ismi varsayarsak ESKİ koşunun
    sonuçlarını okur ve yanlış modeli değerlendiririz — bu yüzden en son
    değiştirilen dizin seçilir.
    """
    base = Path(TRAIN_CONFIG['project'])
    cands = [p for p in base.glob(TRAIN_CONFIG['name'] + '*') if p.is_dir()]
    return max(cands, key=lambda p: p.stat().st_mtime) if cands else base / TRAIN_CONFIG['name']



def dataset_hazirla():
    """dataset/ hazır değilse depodaki hazırlık scriptini çalıştırır.

    Eğitim ve değerlendirme hücreleri bunu kendisi çağırır: 4️⃣ hücresini
    atlasanız veya oturum yenilense bile dataset otomatik hazırlanır.
    Zaten hazırsa saniyeler içinde döner, yeniden açma yapmaz.
    """
    import subprocess, sys
    r = subprocess.run(
        [sys.executable, 'scripts/prepare_colab_dataset.py',
         '--drive-root', str(DRIVE_ROOT), '--repo', str(REPO_DIR)],
        cwd=str(REPO_DIR),
    )
    if r.returncode != 0:
        raise RuntimeError('Dataset hazırlanamadı — yukarıdaki mesaja bakın.')


print('\n--- Eğitim ayarları ---')
for k in ('model', 'epochs', 'batch', 'imgsz', 'workers', 'optimizer',
          'cos_lr', 'amp', 'cache', 'patience', 'save_period'):
    print(f'  {k}: {TRAIN_CONFIG.get(k)}')
print('\n  sonuç dizini:', TRAIN_CONFIG['project'])

## 6️⃣ Eğitim

Hücrenin başındaki **`MOD`** ayarı ne yapılacağını belirler:

| MOD | Ne yapar |
|---|---|
| `'otomatik'` ⭐ | Yarım kalmış eğitim varsa **devam eder**, bitmişse **atlar**, hiç yoksa başlatır |
| `'devam'` | Yarım kalmış eğitimden **devam etmeye zorlar**; yoksa hata verir (sessizce sıfırdan başlamaz) |
| `'sifirdan'` | Mevcut checkpoint'leri **yok sayar**, yepyeni koşu açar |

> ⚠️ **Devam ederken `OVERRIDES` etkisizdir** — Ultralytics ayarları checkpoint'ten
> okur. Ayar değiştirip yeniden eğitmek istiyorsanız `MOD = 'sifirdan'` kullanın.

Hücre çalışınca önce **mevcut koşuların listesini** basar: hangi eğitim kaçıncı
epoch'ta, tamamlanmış mı yarım mı — hepsi görünür.

Her `save_period` epoch'ta checkpoint doğrudan Drive'a yazılır; oturum koparsa
hücreyi tekrar çalıştırmanız yeterlidir.


In [ ]:
# ÖN KOŞUL: Yeni Colab oturumunda paketler ve değişkenler sıfırlanır.
# Bu hücre tek başına çalışmaz; kısa yol: Çalışma zamanı → Öncekileri çalıştır (Run before)
_eksik = [a for a in ('TRAIN_CONFIG', 'DATA_YAML_PATH', 'MODELS_DIR', 'dataset_hazirla')
          if a not in globals()]
if _eksik:
    raise RuntimeError('Önce üstteki hücreleri çalıştırın — eksik: ' + ', '.join(_eksik) +
                       '\nColab menüsü: Çalışma zamanı → Öncekileri çalıştır (Run before)')

try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    raise RuntimeError('ultralytics kurulu değil — 1️⃣ Kurulum hücresini çalıştırın '
                       '(veya Çalışma zamanı → Öncekileri çalıştır).') from None

# Dataset hazır değilse otomatik hazırla (4️⃣ hücresini atlasanız da çalışır)
dataset_hazirla()

import time, shutil
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════════
#  MOD SEÇİMİ  —  ne yapılacağını buradan belirleyin
#
#    'otomatik'  → Yarım kalmış eğitim varsa DEVAM eder, bitmişse ATLAR,
#                  hiç yoksa BAŞTAN başlatır.            (önerilen)
#    'devam'     → Yarım kalmış eğitimden DEVAM etmeye zorlar.
#                  Devam edilecek eğitim yoksa hata verir, sessizce
#                  sıfırdan başlatmaz.
#    'sifirdan'  → Mevcut checkpoint'leri YOK SAYAR, yepyeni bir koşu açar.
#                  OVERRIDES ile ayar değiştirdiyseniz bunu kullanın.
# ═══════════════════════════════════════════════════════════════════════════
MOD = 'otomatik'


def _kosular():
    """project/name* dizinlerini en yeniden eskiye sıralar."""
    base = Path(TRAIN_CONFIG['project'])
    if not base.exists():
        return []
    return sorted([p for p in base.glob(TRAIN_CONFIG['name'] + '*') if p.is_dir()],
                  key=lambda p: p.stat().st_mtime, reverse=True)


def _durum(d):
    """(durum, tamamlanan_epoch) → durum: 'yok' | 'yarim' | 'bitti'"""
    if not (d / 'weights' / 'last.pt').exists():
        return 'yok', 0
    csv = d / 'results.csv'
    n = 0
    if csv.exists():
        try:
            n = max(0, sum(1 for _ in csv.open(encoding='utf-8', errors='ignore')) - 1)
        except OSError:
            n = 0
    return ('bitti' if n >= int(TRAIN_CONFIG['epochs']) else 'yarim'), n


def _devam_edilebilir():
    """Devam edilecek koşuyu döner: EN ÇOK İLERLEMİŞ yarım koşu.

    "En yeni" değil "en ilerlemiş" seçilir; aksi halde yanlışlıkla açılmış
    1 epoch'luk yeni bir koşu, 120 epoch ilerlemiş koşunun önüne geçerdi.
    """
    adaylar = [(d, _durum(d)[1]) for d in _kosular() if _durum(d)[0] == 'yarim']
    if not adaylar:
        return None, 0
    return max(adaylar, key=lambda x: (x[1], x[0].stat().st_mtime))


# Mevcut koşuları göster — hangi eğitimin nerede kaldığı net görünsün
kosular = _kosular()
if kosular:
    print(f'📋 Mevcut koşular ({TRAIN_CONFIG["project"]}):')
    for d in kosular:
        durum, n = _durum(d)
        etiket = {'bitti': '✅ tamamlandı', 'yarim': '⏸️ yarım kaldı', 'yok': '— checkpoint yok'}[durum]
        isaret = '   ⬅ devam edilecek' if d == _devam_edilebilir()[0] else ''
        print(f'   {d.name:<20} {n:>4}/{TRAIN_CONFIG["epochs"]} epoch  {etiket}{isaret}')
    print()

devam_dir, devam_epoch = _devam_edilebilir()
model = None
t0 = time.time()

if MOD == 'devam':
    if not devam_dir:
        raise RuntimeError(
            "MOD='devam' seçildi ama devam edilebilecek yarım eğitim yok.\n"
            "Yukarıdaki listeye bakın; yeni bir eğitim başlatmak için MOD='sifirdan' yapın."
        )
    print(f'🔄 DEVAM: {devam_dir.name} ({devam_epoch}. epoch\'tan sonrası)')
    print("   (ayarlar checkpoint'ten okunur; OVERRIDES bu modda etkisizdir)")
    model = YOLO(str(devam_dir / 'weights' / 'last.pt'))
    model.train(resume=True)

elif MOD == 'sifirdan':
    print('↻ SIFIRDAN: mevcut checkpoint\'ler yok sayılıyor, yeni koşu açılıyor')
    model = YOLO(TRAIN_CONFIG['model'])
    model.train(data=DATA_YAML_PATH, **TRAIN_CONFIG)

elif MOD == 'otomatik':
    bitmis = [d for d in kosular if _durum(d)[0] == 'bitti']
    if devam_dir:
        print(f'🔄 Yarım kalmış eğitim bulundu → devam ediliyor: {devam_dir.name} '
              f'({devam_epoch}. epoch\'tan sonrası)')
        print("   (ayarlar checkpoint'ten okunur; OVERRIDES bu modda etkisizdir)")
        model = YOLO(str(devam_dir / 'weights' / 'last.pt'))
        try:
            model.train(resume=True)
        except Exception as e:
            print(f'⚠️ Devam edilemedi ({type(e).__name__}: {e}) → yeni koşu başlatılıyor')
            model = YOLO(TRAIN_CONFIG['model'])
            model.train(data=DATA_YAML_PATH, **TRAIN_CONFIG)
    elif bitmis:
        RUN_DIR = bitmis[0]
        print(f'✅ Eğitim zaten tamamlanmış: {RUN_DIR}')
        print('   Değerlendirme hücresine geçebilirsiniz.')
        print("   Yeniden eğitmek için: MOD = 'sifirdan'")
    else:
        print('🚀 Eğitim başlıyor (ilk koşu)...')
        model = YOLO(TRAIN_CONFIG['model'])   # yolo26s.pt ilk çalıştırmada indirilir
        model.train(data=DATA_YAML_PATH, **TRAIN_CONFIG)

else:
    raise ValueError(f"MOD geçersiz: {MOD!r} — 'otomatik', 'devam' veya 'sifirdan' olmalı")

if model is not None:
    print(f'\n✅ Eğitim bitti: {(time.time()-t0)/3600:.2f} saat')
    # Çıktı dizinini VARSAYMA, trainer'dan al
    RUN_DIR = Path(model.trainer.save_dir)

print('📊 Sonuçlar:', RUN_DIR)
best_path = RUN_DIR / 'weights' / 'best.pt'
if best_path.exists():
    dest = MODELS_DIR / f'best_{RUN_DIR.name}.pt'
    shutil.copy(best_path, dest)
    print("🏆 En iyi model Drive'a kopyalandı:", dest)
else:
    print('⚠️ best.pt yok — eğitim loglarını kontrol edin.')


## 7️⃣ Değerlendirme — sınıf bazlı (ticari karar buradan verilir)

Genel mAP tek başına yanıltıcıdır: ortalama iyi görünürken tek bir hastalıkta recall
çok düşük olabilir. Az örnekli sınıflara (Anthracnose Fruit Rot, Powdery Mildew Fruit)
ayrıca bakın.

In [ ]:
# ÖN KOŞUL: Yeni Colab oturumunda paketler ve değişkenler sıfırlanır.
# Bu hücre tek başına çalışmaz; kısa yol: Çalışma zamanı → Öncekileri çalıştır (Run before)
_eksik = [a for a in ('DATA_YAML_PATH', 'find_run_dir', 'dataset_hazirla') if a not in globals()]
if _eksik:
    raise RuntimeError('Önce üstteki hücreleri çalıştırın — eksik: ' + ', '.join(_eksik) +
                       '\nColab menüsü: Çalışma zamanı → Öncekileri çalıştır (Run before)')

try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    raise RuntimeError('ultralytics kurulu değil — 1️⃣ Kurulum hücresini çalıştırın '
                       '(veya Çalışma zamanı → Öncekileri çalıştır).') from None

# Dataset hazır değilse otomatik hazırla (4️⃣ hücresini atlasanız da çalışır)
dataset_hazirla()

from pathlib import Path

# Koşu dizini: eğitim bu oturumda yapıldıysa gerçek save_dir, değilse en yeni koşu
run_dir = RUN_DIR if 'RUN_DIR' in globals() else find_run_dir()
best_path = run_dir / 'weights' / 'best.pt'
print('📂 Koşu dizini:', run_dir)

if not best_path.exists():
    print('⚠️ best.pt bulunamadı — önce eğitim hücresini çalıştırın.')
else:
    model = YOLO(str(best_path))
    m = model.val(data=DATA_YAML_PATH)   # eğitimdekiyle AYNI config

    print('\n' + '='*58)
    print(f'GENEL  mAP50: {m.box.map50:.4f} | mAP50-95: {m.box.map:.4f} | '
          f'P: {m.box.mp:.4f} | R: {m.box.mr:.4f}')
    print('='*58)
    print(f"\n{'Sınıf':<24}{'P':>8}{'R':>8}{'mAP50':>9}")
    cls_names = model.names
    zayif = []
    for i, c in enumerate(m.box.ap_class_index):
        r = float(m.box.r[i])
        flag = '  ⚠️ düşük recall' if r < 0.75 else ''
        if r < 0.75:
            zayif.append(cls_names[int(c)])
        print(f'{cls_names[int(c)]:<24}{m.box.p[i]:>8.3f}{r:>8.3f}{m.box.ap50[i]:>9.3f}{flag}')

    if zayif:
        print(f'\n⚠️ Recall < 0.75 olan sınıflar: {", ".join(zayif)}')
        print('   → Bu sınıflar için augmentasyon çarpanını artırmak yerine GERÇEK veri toplayın.')
    print('\n💡 confusion_matrix.png: hangi hastalık hangisiyle karışıyor?')

In [ ]:
# Eğitim grafikleri ve confusion matrix
from IPython.display import Image, display
from pathlib import Path

rd = RUN_DIR if 'RUN_DIR' in globals() else find_run_dir()
print('📂 Koşu dizini:', rd)
for f in ('results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg'):
    p = rd / f
    if p.exists():
        print(f)
        display(Image(filename=str(p)))

## 8️⃣ Örnek tahminler

Yüksek çözünürlüklü saha fotoğraflarında küçük lezyonlar için
`scripts/sahi_predict.py` (dilimli inference) kullanın.

In [ ]:
# ÖN KOŞUL: Yeni Colab oturumunda paketler ve değişkenler sıfırlanır.
# Bu hücre tek başına çalışmaz; kısa yol: Çalışma zamanı → Öncekileri çalıştır (Run before)
_eksik = [a for a in ('DATA_YAML_PATH', 'find_run_dir', 'dataset_hazirla') if a not in globals()]
if _eksik:
    raise RuntimeError('Önce üstteki hücreleri çalıştırın — eksik: ' + ', '.join(_eksik) +
                       '\nColab menüsü: Çalışma zamanı → Öncekileri çalıştır (Run before)')

try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    raise RuntimeError('ultralytics kurulu değil — 1️⃣ Kurulum hücresini çalıştırın '
                       '(veya Çalışma zamanı → Öncekileri çalıştır).') from None

# Dataset hazır değilse otomatik hazırla (4️⃣ hücresini atlasanız da çalışır)
dataset_hazirla()

import cv2, yaml
import matplotlib.pyplot as plt
from pathlib import Path

cfg = yaml.safe_load(Path(DATA_YAML_PATH).read_text(encoding='utf-8'))
root = Path(cfg.get('path') or Path(DATA_YAML_PATH).parent)
val_dirs = cfg['val'] if isinstance(cfg['val'], list) else [cfg['val']]

imgs = []
for d in val_dirs:
    p = (root / d).resolve()
    if p.exists():
        imgs += sorted(p.glob('*.jpg'))[:2]

best_path = (RUN_DIR if 'RUN_DIR' in globals() else find_run_dir()) / 'weights' / 'best.pt'
if imgs and best_path.exists():
    model = YOLO(str(best_path))
    for ip in imgs[:5]:
        r = model(str(ip), verbose=False)[0]
        plt.figure(figsize=(11, 7))
        plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off')
        plt.title(ip.name); plt.show()
        counts = {}
        for b in r.boxes:
            n = model.names[int(b.cls[0])]
            counts[n] = counts.get(n, 0) + 1
        print(f"{ip.name}: {len(r.boxes)} tespit → {counts or 'yok'}\n" + '-'*50)
else:
    print('⚠️ Görüntü veya model bulunamadı.')

---
## 📝 Notlar

**Sonuçlar nerede?** Eğitim doğrudan Drive'a yazar:
```
MyDrive/SmartFarmStrawberryDisease/
├── results/strawberry_exp/        # grafikler, confusion matrix, weights/
└── best_models/best_strawberry_exp.pt
```

**Colab Pro+ ipuçları**

| Ayar | Nerede | Etkisi |
|---|---|---|
| **A100 GPU** | Runtime → Change runtime type | En hızlı; 200 epoch ~3-4 saat (T4'te 15+ saat) |
| **High-RAM** | Runtime → Change runtime type | A100'de `cache='ram'` açılır, disk darboğazı kalkar |
| **Background execution** | Runtime menüsü | Tarayıcı kapalıyken de eğitim sürer |

GPU tipi **koddan seçilemez** — runtime ayarıdır. Kod, hangi GPU verildiyse
`batch`/`workers`/`cache` değerlerini ona göre otomatik ayarlar.

**Sık karşılaşılan sorunlar**

| Sorun | Çözüm |
|---|---|
| `images not found` | `data=` mutlak yol mu? (bu notebook otomatik yapar) |
| CUDA out of memory | 5️⃣ hücresinde `OVERRIDES = {'batch': <yarısı>}` |
| RAM doldu / oturum çöktü | `OVERRIDES = {'cache': False}` (High-RAM kapalıysa) |
| `numpy.dtype size changed` / cv2 import hatası | Runtime → Restart session, sonra 1️⃣ hücresi |
| `yolo26s.pt` yüklenemiyor | `!pip install -U ultralytics` (>=8.3.200 gerekir) |
| Oturum koptu | 6️⃣ bölümündeki "devam et" hücresi |

**Dataset güncellenirse:** Bilgisayarda yeni `dataset_colab.zip` oluşturup Drive'daki
dosyanın üzerine yazın, Colab'da `dataset/` klasörünü silip 4️⃣ hücresini tekrar çalıştırın.